# Feature Engineering — Readmission Risk Prediction

Aggregates clinical vitals per admission and joins with admission attributes
to build a per-admission feature table for 30-day readmission classification.

**Reads:** `silver_patient_admissions`, `silver_clinical_records`  **Writes:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, hour, count, avg,
    sum as spark_sum
)

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
adm = spark.read.table('silver_patient_admissions')
clin = spark.read.table('silver_clinical_records')
print(f'Admissions: {adm.count():,} | Clinical records: {clin.count():,}')

required = {'admission_id', 'is_readmission', 'length_of_stay_days'}
missing = required - set(adm.columns)
if missing:
    raise ValueError(f'silver_patient_admissions missing columns {sorted(missing)}. Regenerate data and rerun bronze/silver.')

In [ ]:
# Aggregate vitals per admission
vitals = (
    clin.groupBy('admission_id')
    .agg(
        count('*').alias('vital_count'),
        spark_sum(when(col('is_abnormal'), 1).otherwise(0)).alias('abnormal_vital_count'),
        avg('value').alias('avg_vital_value'),
    )
    .withColumn('abnormal_vital_ratio',
        col('abnormal_vital_count') / when(col('vital_count') > 0, col('vital_count')).otherwise(1))
)

In [ ]:
# Join admission attributes + vitals. EXCLUDE any readmission-derived columns.
ml_features = (
    adm.select(
        'admission_id', 'patient_id', 'department', 'admission_type',
        'insurance_type', 'age_group', 'dx_chapter', 'los_bucket',
        'length_of_stay_days', 'prior_admissions',
        hour('admission_date').alias('admission_hour'),
        col('is_readmission').cast('int').alias('had_readmission'),
    )
    .join(vitals, 'admission_id', 'left')
    .na.fill(0, ['vital_count', 'abnormal_vital_count', 'avg_vital_value', 'abnormal_vital_ratio'])
    .na.fill('unknown', subset=['department', 'admission_type', 'insurance_type', 'age_group', 'dx_chapter'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_readmission') == 1).count()
readmit_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

if total_rows < 1000 or positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} readmission rows '
        f'({readmit_rate:.2f}%). Check is_readmission typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')
print(f'Gold ML features written: {total_rows:,} rows | readmission rate {readmit_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')